In [2]:
from pathlib import Path
import sys

# set paths
SCRIPT_DIR = Path.cwd()
sys.path.insert(0, str(SCRIPT_DIR))
ROOT = SCRIPT_DIR.parent
RAW_DIR = ROOT / "data" #directory with the timeseries
RUN_LIST_PATH = ROOT / "info" / "onerun.csv" #list of runs you want

# global vars
RANDOM_SEED = 20260429
PERMUTATION_DRAWS = 100000

In [3]:
# make the manifest from run list

import pandas as pd
from lib_paths import AA_MANIFEST
metadata = pd.read_csv(RUN_LIST_PATH, dtype=str).fillna("")
metadata.columns = [column.strip().lower() for column in metadata.columns]
metadata["run"] = metadata["trial"].str.strip()
metadata["raw_csv"] = metadata["run"].map(lambda run: f"{run}.csv")
metadata[["run", "raw_csv", "caught", "fence", "lure"]].to_csv(AA_MANIFEST, index=False)

In [4]:
# make the main dataframe including all derived kinematics (velocity, angles, etc.)

import pandas as pd
from lib_samples import prepare_run
from lib_paths import AA_FILTERS, AA_MANIFEST, AA_SAMPLES

manifest = pd.read_csv(AA_MANIFEST)

sample_tables = []
filter_rows = []
for _, row in manifest.iterrows():
    samples, filter_row = prepare_run(row, RAW_DIR, len(manifest))
    sample_tables.append(samples)
    filter_rows.append(filter_row)

pd.concat(sample_tables, ignore_index=True).to_csv(AA_SAMPLES, index=False)
pd.DataFrame(filter_rows).to_csv(AA_FILTERS, index=False)

In [5]:
# compute power spectra and stride-scale time windows

import numpy as np
import pandas as pd
from scipy.signal import welch
from lib_paths import AA_SAMPLES, BB_PSD, BB_STRIDES

PSD_VARIABLES = ["a_parallel", "a_perp"]
STRIDE_MIN_HZ = 1.0
STRIDE_MAX_HZ = 6.0
NPERSEG = 256

samples = pd.read_csv(AA_SAMPLES).sort_values(["run", "t"]).reset_index(drop=True)

psd_rows = []
stride_rows = []
for run, run_df in samples.groupby("run", sort=True):
    sample_rate_hz = float(run_df["sample_rate_hz"].iloc[0])
    nperseg = min(NPERSEG, len(run_df))
    noverlap = nperseg // 2
    run_psd = {}
    for variable in PSD_VARIABLES:
        x = run_df[variable].to_numpy(dtype=float)
        freq_hz, psd = welch(x - x.mean(), fs=sample_rate_hz, window="hann", nperseg=nperseg, noverlap=noverlap, detrend=False, scaling="density")
        run_psd[variable] = psd
        for freq, value in zip(freq_hz, psd):
            psd_rows.append({"run": run, "variable": variable, "freq_hz": float(freq), "psd": float(value)})

    band = (freq_hz >= STRIDE_MIN_HZ) & (freq_hz <= STRIDE_MAX_HZ)
    score = run_psd["a_parallel"][band] + run_psd["a_perp"][band]
    stride_freq_hz = float(freq_hz[band][np.argmax(score)])
    stride_period_s = 1.0 / stride_freq_hz
    stride_rows.append(
        {
            "run": run,
            "sample_rate_hz": sample_rate_hz,
            "stride_freq_hz": stride_freq_hz,
            "stride_period_s": stride_period_s,
            "stride_frames": max(1, int(round(stride_period_s * sample_rate_hz))),
        }
    )

pd.DataFrame(psd_rows).to_csv(BB_PSD, index=False)
pd.DataFrame(stride_rows).to_csv(BB_STRIDES, index=False)
pd.DataFrame(stride_rows)


,run,sample_rate_hz,stride_freq_hz,stride_period_s,stride_frames
0,29May2025_Mashika,60.0,2.34375,0.426667,26


In [16]:
# stride windows

import numpy as np
import pandas as pd

from lib_paths import AA_SAMPLES, BB_STRIDES, BB_WINDOWS


samples = pd.read_csv(AA_SAMPLES).sort_values(["run", "t"]).reset_index(drop=True)
strides = pd.read_csv(BB_STRIDES).set_index("run")
rows = []

for run, run_df in samples.groupby("run", sort=True):
    run_df = run_df.reset_index(drop=True)
    stride_frames = int(strides.loc[run, "stride_frames"])
    for window_index, start in enumerate(range(0, len(run_df) - stride_frames + 1, stride_frames)):
        window = run_df.iloc[start : start + stride_frames]
        a_parallel = float(window["a_parallel"].mean())
        a_perp = float(window["a_perp"].mean())
        speed = float(window["speed_L"].mean())
        lateral_abs = abs(a_perp)
        psi = np.unwrap(window["psi_L"].to_numpy(dtype=float))
        rows.append(
            {
                "run": run,
                "window_index": window_index,
                "t_start": float(window["t"].iloc[0]),
                "t_end": float(window["t"].iloc[-1]),
                "n_samples": int(len(window)),
                "speed_mps": speed,
                "a_parallel_mps2": a_parallel,
                "a_perp_mps2": a_perp,
                "lateral_abs_mps2": lateral_abs,
                "propulsion_mps2": max(a_parallel, 0.0),
                "braking_mps2": max(-a_parallel, 0.0),
                "heading_change_deg": float(abs(psi[-1] - psi[0]) * 180.0 / np.pi),
                "turn_radius_m": float(speed * speed / lateral_abs) if lateral_abs >= 0.5 else np.nan,
            }
        )

pd.DataFrame(rows).to_csv(BB_WINDOWS, index=False)

#pd.DataFrame(rows).head(5)

In [15]:
# quantiles of frame-wise kinematic quantities

import numpy as np
import pandas as pd
from lib_paths import AA_SAMPLES


def weighted_quantile(values, weights, q):
    order = np.argsort(values)
    x = values[order]
    w = weights[order]
    p = (np.cumsum(w) - 0.5 * w) / np.sum(w)
    return float(np.interp(q, p, x))


samples = pd.read_csv(AA_SAMPLES)
samples["abs_a_parallel"] = np.abs(samples["a_parallel"].to_numpy(dtype=float))
samples["abs_a_perp"] = np.abs(samples["a_perp"].to_numpy(dtype=float))
weights = samples["frame_weight"].to_numpy(dtype=float)

quantiles = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.98, 0.99]

for variable in ["speed_L", "abs_a_parallel", "abs_a_perp", "R"]:
    rows = []
    values = samples[variable].to_numpy(dtype=float)
    for q in quantiles:
        rows.append({"variable": variable, "quantile": q, "value": weighted_quantile(values, weights, q)})
    print(pd.DataFrame(rows))


  variable  quantile      value
0  speed_L      0.10   5.790310
1  speed_L      0.25   6.837043
2  speed_L      0.50   8.081162
3  speed_L      0.75   8.747346
4  speed_L      0.90   9.430263
5  speed_L      0.95   9.963625
6  speed_L      0.98  10.645998
7  speed_L      0.99  10.791918
         variable  quantile      value
0  abs_a_parallel      0.10   0.081366
1  abs_a_parallel      0.25   0.283641
2  abs_a_parallel      0.50   1.055268
3  abs_a_parallel      0.75   4.313146
4  abs_a_parallel      0.90  12.091575
5  abs_a_parallel      0.95  16.693191
6  abs_a_parallel      0.98  20.606693
7  abs_a_parallel      0.99  28.810898
     variable  quantile      value
0  abs_a_perp      0.10   0.095459
1  abs_a_perp      0.25   0.270356
2  abs_a_perp      0.50   0.951599
3  abs_a_perp      0.75   3.933577
4  abs_a_perp      0.90   9.190455
5  abs_a_perp      0.95  12.476800
6  abs_a_perp      0.98  15.675205
7  abs_a_perp      0.99  16.787805
  variable  quantile      value
0        R    

In [18]:
# quantiles of window-averaged kinematic quantities

import numpy as np
import pandas as pd

from lib_paths import BB_WINDOWS


windows = pd.read_csv(BB_WINDOWS)
quantiles = [0.90, 0.98]
variables = ["speed_mps", "lateral_abs_mps2", "propulsion_mps2", "braking_mps2"]
rows = []

for variable in variables:
    for q in quantiles:
        run_values = []
        for _, run_df in windows.groupby("run", sort=True):
            values = run_df[variable].to_numpy(dtype=float)
            run_values.append(float(np.quantile(values, q)))
        rows.append(
            {
                "variable": variable,
                "quantile": q,
                "run_q25": float(np.quantile(run_values, 0.25)),
                "run_median": float(np.median(run_values)),
                "run_q75": float(np.quantile(run_values, 0.75)),
                "n_runs": int(len(run_values)),
            }
        )

print(pd.DataFrame(rows))

           variable  quantile    run_q25  run_median    run_q75  n_runs
0         speed_mps      0.90   9.077494    9.077494   9.077494       1
1         speed_mps      0.98  10.120064   10.120064  10.120064       1
2  lateral_abs_mps2      0.90   3.758644    3.758644   3.758644       1
3  lateral_abs_mps2      0.98   5.281638    5.281638   5.281638       1
4   propulsion_mps2      0.90   2.475679    2.475679   2.475679       1
5   propulsion_mps2      0.98   4.683041    4.683041   4.683041       1
6      braking_mps2      0.90   2.944665    2.944665   2.944665       1
7      braking_mps2      0.98   4.430004    4.430004   4.430004       1


In [19]:
# bin window-averaged quantities by speed

import numpy as np
import pandas as pd
from lib_paths import BB_WINDOWS, CC_SPEED_BINS

windows = pd.read_csv(BB_WINDOWS)
edges = np.linspace(0.0, float(windows["speed_mps"].quantile(0.98)), 11)
variables = ["lateral_abs_mps2", "propulsion_mps2", "braking_mps2"]
rows = []

for variable in variables:
    for bin_index in range(len(edges) - 1):
        left = edges[bin_index]
        right = edges[bin_index + 1]
        for run, run_df in windows.groupby("run", sort=True):
            if bin_index == len(edges) - 2:
                mask = (run_df["speed_mps"] >= left) & (run_df["speed_mps"] <= right)
            else:
                mask = (run_df["speed_mps"] >= left) & (run_df["speed_mps"] < right)
            if mask.any():
                values = run_df.loc[mask, variable].to_numpy(dtype=float)
                rows.append(
                    {
                        "variable": variable,
                        "run": run,
                        "bin_index": bin_index,
                        "speed_left": float(left),
                        "speed_right": float(right),
                        "speed_mid": float(0.5 * (left + right)),
                        "median": float(np.median(values)),
                        "q98": float(np.quantile(values, 0.98)),
                        "n_windows": int(len(values)),
                    }
                )

pd.DataFrame(rows).to_csv(CC_SPEED_BINS, index=False)


In [21]:
# line-of-sight variables run-wise summaries

import numpy as np
import pandas as pd
from lib_paths import AA_SAMPLES

samples = pd.read_csv(AA_SAMPLES)
samples["abs_theta"] = np.abs(samples["theta"].to_numpy(dtype=float))
samples["abs_lambda_dot"] = np.abs(samples["lambda_dot"].to_numpy(dtype=float))

rows = []
for variable in ["R", "theta", "abs_theta", "lambda_dot", "abs_lambda_dot", "R_dot"]:
    for run, run_df in samples.groupby("run", sort=True):
        values = run_df[variable].to_numpy(dtype=float)
        rows.append(
            {
                "variable": variable,
                "run": run,
                "q10": float(np.quantile(values, 0.10)),
                #"q25": float(np.quantile(values, 0.25)),
                "q50": float(np.quantile(values, 0.50)),
                #"q75": float(np.quantile(values, 0.75)),
                "q90": float(np.quantile(values, 0.90)),
            }
        )

print(pd.DataFrame(rows))

         variable                run       q10       q50        q90
0               R  29May2025_Mashika  4.417157  8.452354  20.229633
1           theta  29May2025_Mashika -0.571817 -0.224868  -0.093292
2       abs_theta  29May2025_Mashika  0.093292  0.224868   0.571817
3      lambda_dot  29May2025_Mashika -0.797591 -0.045177   0.272835
4  abs_lambda_dot  29May2025_Mashika  0.038032  0.205980   0.797591
5           R_dot  29May2025_Mashika -7.327534 -1.824716   2.392956


In [22]:
# steering models parameter search
# LONG RUN TIME

import pandas as pd

from lib_paths import EE_EVALUATIONS
from lib_steering_fit import fit_model
from lib_steering_model import MODEL_SPECS
from lib_steering_runs import load_runs


runs = load_runs()
evaluations = pd.concat([fit_model(runs, spec) for spec in MODEL_SPECS], ignore_index=True)
evaluations.to_csv(EE_EVALUATIONS, index=False)


In [23]:
# steering models selected steering fits

import pandas as pd

from lib_paths import EE_EVALUATIONS
from lib_steering_fit import selected_summary


selected_summary(pd.read_csv(EE_EVALUATIONS))


,model_key,model_label,stage,T_s,K,N,run_equal_mean_error_m,median_run_mean_error_m,q25_run_median_error_m,median_run_median_error_m,q75_run_median_error_m
0,pn,PN,refine_2,NaN,0.000000,1.839000,2.010782,2.010782,1.379906,1.379906,1.379906
1,pp,PP,grid,NaN,1.000000,0.000000,1.876593,1.876593,1.465188,1.465188,1.465188
2,pp_pn,PP+PN,refine_2,4.0,0.700804,2.803214,2.328878,2.328878,1.694980,1.694980,1.694980


In [24]:
# steering run errors

import numpy as np
import pandas as pd

from lib_paths import EE_EVALUATIONS, EE_RUN_ERRORS
from lib_steering_fit import selected_summary
from lib_steering_model import MODEL_SPECS
from lib_steering_runs import load_runs, run_errors


summary = selected_summary(pd.read_csv(EE_EVALUATIONS))
runs = load_runs()
specs = {spec["model_key"]: spec for spec in MODEL_SPECS}
rows = []

for _, fit in summary.iterrows():
    spec = specs[fit["model_key"]]
    params = np.array([[float(fit[name]) for name in spec["param_names"]]], dtype=float)
    for run in runs:
        mean_error, median_error = run_errors(run, spec, params)
        rows.append(
            {
                "model_key": fit["model_key"],
                "model_label": fit["model_label"],
                "run": run["run"],
                "mean_euclidean_error_m": float(mean_error[0]),
                "median_euclidean_error_m": float(median_error[0]),
            }
        )

pd.DataFrame(rows).to_csv(EE_RUN_ERRORS, index=False)


In [25]:
# steering models pairwise comparison tests

import numpy as np
import pandas as pd
from scipy import stats

from lib_math import holm_adjust, sign_flip_pvalue
from lib_paths import EE_RUN_ERRORS


comparisons = [("pp_pn", "pp"), ("pp_pn", "pn"), ("pp", "pn")]
errors = pd.read_csv(EE_RUN_ERRORS)
rng = np.random.default_rng(RANDOM_SEED + 1)
rows = []

for first, second in comparisons:
    a = errors.loc[errors["model_key"] == first].set_index("run")
    b = errors.loc[errors["model_key"] == second].set_index("run")
    diff = b.loc[a.index, "median_euclidean_error_m"].to_numpy(dtype=float) - a["median_euclidean_error_m"].to_numpy(dtype=float)
    rows.append(
        {
            "comparison": f"{first}_vs_{second}",
            "first_model": first,
            "second_model": second,
            "mean_difference_m": float(diff.mean()),
            "median_difference_m": float(np.median(diff)),
            "n_runs": int(len(diff)),
            "first_wins": int(np.count_nonzero(diff > 0)),
            "second_wins": int(np.count_nonzero(diff < 0)),
            "wilcoxon_p": float(stats.wilcoxon(diff[diff != 0.0], alternative="two-sided").pvalue),
            "permutation_p": sign_flip_pvalue(diff, PERMUTATION_DRAWS, rng),
        }
    )

tests = pd.DataFrame(rows)
tests["wilcoxon_p_holm"] = holm_adjust(tests["wilcoxon_p"].to_numpy(dtype=float))
tests["permutation_p_holm"] = holm_adjust(tests["permutation_p"].to_numpy(dtype=float))
tests


,comparison,first_model,second_model,mean_difference_m,median_difference_m,n_runs,first_wins,second_wins,wilcoxon_p,permutation_p,wilcoxon_p_holm,permutation_p_holm
0,pp_pn_vs_pp,pp_pn,pp,-0.229793,-0.229793,1,0,1,1.0,1.0,1.0,1.0
1,pp_pn_vs_pn,pp_pn,pn,-0.315075,-0.315075,1,0,1,1.0,1.0,1.0,1.0
2,pp_vs_pn,pp,pn,-0.085282,-0.085282,1,0,1,1.0,1.0,1.0,1.0


In [26]:
# steering parameter basins

import numpy as np
import pandas as pd

from lib_grid import smooth_values
from lib_paths import EE_EVALUATIONS
from lib_steering_model import BASIN_MARGIN_M, MODEL_SPECS, OBJECTIVE


evaluations = pd.read_csv(EE_EVALUATIONS)
rows = []

for spec in MODEL_SPECS:
    grid = evaluations.loc[(evaluations["model_key"] == spec["model_key"]) & (evaluations["stage"] == "grid")].copy()
    params = list(spec["param_names"])
    if len(params) == 1:
        param = params[0]
        grid = grid.sort_values(param).reset_index(drop=True)
        smooth = smooth_values(grid[OBJECTIVE].to_numpy(dtype=float), 1)
        mask = smooth <= smooth.min() + BASIN_MARGIN_M
        start = None
        basin_rank = 1
        for i, keep in enumerate(mask.tolist() + [False]):
            if keep and start is None:
                start = i
            if not keep and start is not None:
                stop = i
                if stop - start >= 5:
                    sub = grid.iloc[start:stop]
                    rows.append({"model_key": spec["model_key"], "basin_rank": basin_rank, "n_grid_cells": int(len(sub)), f"{param}_min": float(sub[param].min()), f"{param}_max": float(sub[param].max()), "best_error_m": float(sub[OBJECTIVE].min())})
                    basin_rank += 1
                start = None
    else:
        heat = grid.pivot_table(index="N", columns="T_s", values=OBJECTIVE, aggfunc="min").sort_index().sort_index(axis=1)
        values = heat.to_numpy(dtype=float)
        mask = values <= values.min() + BASIN_MARGIN_M
        seen = np.zeros(mask.shape, dtype=bool)
        basin_rank = 1
        for r in range(mask.shape[0]):
            for c in range(mask.shape[1]):
                if seen[r, c] or not mask[r, c]:
                    continue
                stack = [(r, c)]
                cells = []
                seen[r, c] = True
                while stack:
                    rr, cc = stack.pop()
                    cells.append((rr, cc))
                    for nr, nc in [(rr - 1, cc), (rr + 1, cc), (rr, cc - 1), (rr, cc + 1)]:
                        if 0 <= nr < mask.shape[0] and 0 <= nc < mask.shape[1] and mask[nr, nc] and not seen[nr, nc]:
                            seen[nr, nc] = True
                            stack.append((nr, nc))
                if len(cells) >= 8:
                    n_vals = [float(heat.index[rr]) for rr, _ in cells]
                    t_vals = [float(heat.columns[cc]) for _, cc in cells]
                    errors = [float(values[rr, cc]) for rr, cc in cells]
                    rows.append({"model_key": spec["model_key"], "basin_rank": basin_rank, "n_grid_cells": int(len(cells)), "T_s_min": min(t_vals), "T_s_max": max(t_vals), "N_min": min(n_vals), "N_max": max(n_vals), "best_error_m": min(errors)})
                    basin_rank += 1

pd.DataFrame(rows)


,model_key,basin_rank,n_grid_cells,N_min,N_max,best_error_m,T_s_min,T_s_max
0,pn,1,10,1.610000,1.88,1.380913,NaN,NaN
1,pp_pn,1,142,2.321429,3.00,1.696208,3.387143,4.0


In [15]:
# velocity parameter search

import numpy as np
import pandas as pd

from lib_paths import FF_EVALUATIONS
from lib_velocity_anchors import nested_anchors
from lib_velocity_fit import fit_model
from lib_velocity_model import MODEL_SPECS
from lib_velocity_runs import load_runs


rng = np.random.default_rng(RANDOM_SEED)
runs = load_runs()
previous_best = []
frames = []

for spec in MODEL_SPECS:
    evaluations = fit_model(runs, spec, rng, nested_anchors(spec, previous_best))
    frames.append(evaluations)
    previous_best.append(evaluations.sort_values("median_run_mean_speed_mae_mps").iloc[0])

pd.concat(frames, ignore_index=True).to_csv(FF_EVALUATIONS, index=False)


In [16]:
# selected velocity fits

import pandas as pd

from lib_paths import FF_EVALUATIONS
from lib_velocity_fit import selected_summary


selected_summary(pd.read_csv(FF_EVALUATIONS))


,model_key,model_label,stage,run_equal_mean_speed_mae_mps,q25_run_mean_speed_mae_mps,median_run_mean_speed_mae_mps,q75_run_mean_speed_mae_mps,median_run_median_speed_ae_mps,k,vstar,bpsi,bR_linear,bR_gaussian,bR_switch,s,t
0,m4_switch,M4 lateral + switch R,refine_2,0.838019,0.838019,0.838019,0.838019,0.676850,1.504772,10.057797,-1.145739,NaN,NaN,2.899948,3.651548,4.553763
1,m3_gaussian,M3 lateral + Gaussian R,refine_2,0.963859,0.963859,0.963859,0.963859,0.887418,0.563624,14.298719,-1.943113,NaN,-1.919146,NaN,7.868240,NaN
2,m2_linear,M2 lateral + linear R,refine_2,0.984624,0.984624,0.984624,0.984624,0.925345,0.465738,12.579114,-1.969245,0.103861,NaN,NaN,NaN,NaN
3,m1_lateral,M1 lateral demand,refine_2,0.993479,0.993479,0.993479,0.993479,0.857186,0.635524,13.816752,-1.935188,NaN,NaN,NaN,NaN,NaN
4,m0_constant,M0 constant target,refine_2,1.366865,1.366865,1.366865,1.366865,0.911934,5.000000,8.112237,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# velocity run errors

import numpy as np
import pandas as pd

from lib_paths import FF_EVALUATIONS, FF_RUN_ERRORS
from lib_velocity_fit import selected_summary
from lib_velocity_model import MODEL_SPECS
from lib_velocity_runs import load_runs, run_metrics


summary = selected_summary(pd.read_csv(FF_EVALUATIONS))
runs = load_runs()
specs = {spec["model_key"]: spec for spec in MODEL_SPECS}
rows = []

for _, fit in summary.iterrows():
    spec = specs[fit["model_key"]]
    params = np.array([[float(fit[name]) for name in spec["param_names"]]], dtype=float)
    for run in runs:
        mean_error, median_error = run_metrics(run, spec, params)
        rows.append(
            {
                "model_key": fit["model_key"],
                "model_label": fit["model_label"],
                "run": run["run"],
                "mean_abs_speed_error_mps": float(mean_error[0]),
                "median_abs_speed_error_mps": float(median_error[0]),
            }
        )

pd.DataFrame(rows).to_csv(FF_RUN_ERRORS, index=False)


In [18]:
# velocity model tests

import numpy as np
import pandas as pd
from scipy import stats

from lib_math import holm_adjust, sign_flip_pvalue
from lib_paths import FF_RUN_ERRORS


comparisons = [("m1_lateral", "m0_constant"), ("m2_linear", "m1_lateral"), ("m3_gaussian", "m1_lateral"), ("m4_switch", "m1_lateral"), ("m4_switch", "m2_linear"), ("m4_switch", "m3_gaussian")]
errors = pd.read_csv(FF_RUN_ERRORS)
rng = np.random.default_rng(RANDOM_SEED + 2)
rows = []

for first, second in comparisons:
    a = errors.loc[errors["model_key"] == first].set_index("run")
    b = errors.loc[errors["model_key"] == second].set_index("run")
    diff = b.loc[a.index, "mean_abs_speed_error_mps"].to_numpy(dtype=float) - a["mean_abs_speed_error_mps"].to_numpy(dtype=float)
    rows.append(
        {
            "comparison": f"{first}_vs_{second}",
            "first_model": first,
            "second_model": second,
            "mean_difference_mps": float(diff.mean()),
            "median_difference_mps": float(np.median(diff)),
            "n_runs": int(len(diff)),
            "first_wins": int(np.count_nonzero(diff > 0)),
            "second_wins": int(np.count_nonzero(diff < 0)),
            "wilcoxon_p": float(stats.wilcoxon(diff[diff != 0.0], alternative="two-sided").pvalue),
            "permutation_p": sign_flip_pvalue(diff, PERMUTATION_DRAWS, rng),
        }
    )

tests = pd.DataFrame(rows)
tests["wilcoxon_p_holm"] = holm_adjust(tests["wilcoxon_p"].to_numpy(dtype=float))
tests["permutation_p_holm"] = holm_adjust(tests["permutation_p"].to_numpy(dtype=float))
tests


,comparison,first_model,second_model,mean_difference_mps,median_difference_mps,n_runs,first_wins,second_wins,wilcoxon_p,permutation_p,wilcoxon_p_holm,permutation_p_holm
0,m1_lateral_vs_m0_constant,m1_lateral,m0_constant,0.373386,0.373386,1,1,0,1.0,1.0,1.0,1.0
1,m2_linear_vs_m1_lateral,m2_linear,m1_lateral,0.008855,0.008855,1,1,0,1.0,1.0,1.0,1.0
2,m3_gaussian_vs_m1_lateral,m3_gaussian,m1_lateral,0.029620,0.029620,1,1,0,1.0,1.0,1.0,1.0
3,m4_switch_vs_m1_lateral,m4_switch,m1_lateral,0.155460,0.155460,1,1,0,1.0,1.0,1.0,1.0
4,m4_switch_vs_m2_linear,m4_switch,m2_linear,0.146605,0.146605,1,1,0,1.0,1.0,1.0,1.0
5,m4_switch_vs_m3_gaussian,m4_switch,m3_gaussian,0.125840,0.125840,1,1,0,1.0,1.0,1.0,1.0


In [19]:
# velocity parameter basins

import numpy as np
import pandas as pd

from lib_paths import FF_EVALUATIONS
from lib_velocity_fit import evaluate_params
from lib_velocity_fit import selected_summary
from lib_velocity_model import MODEL_SPECS, OBJECTIVE, PROFILE_MARGIN_MPS, PROFILE_POINTS
from lib_velocity_runs import load_runs


summary = selected_summary(pd.read_csv(FF_EVALUATIONS))
runs = load_runs()
specs = {spec["model_key"]: spec for spec in MODEL_SPECS}
rows = []

for _, fit in summary.iterrows():
    spec = specs[fit["model_key"]]
    base = np.array([float(fit[name]) for name in spec["param_names"]], dtype=float)
    for param_index, param in enumerate(spec["param_names"]):
        lo, hi = spec["bounds"][param_index]
        params = np.tile(base, (PROFILE_POINTS, 1))
        params[:, param_index] = np.linspace(lo, hi, PROFILE_POINTS)
        profile = evaluate_params(runs, spec, params, "profile").sort_values(param)
        values = profile[OBJECTIVE].to_numpy(dtype=float)
        mask = values <= values.min() + PROFILE_MARGIN_MPS
        if mask.any():
            rows.append(
                {
                    "model_key": spec["model_key"],
                    "parameter": param,
                    "selected_value": float(base[param_index]),
                    "basin_min": float(profile.loc[mask, param].min()),
                    "basin_max": float(profile.loc[mask, param].max()),
                    "profile_min_error_mps": float(values.min()),
                    "n_profile_points": int(mask.sum()),
                }
            )

pd.DataFrame(rows)


,model_key,parameter,selected_value,basin_min,basin_max,profile_min_error_mps,n_profile_points
0,m4_switch,k,1.504772,1.023188,2.514493,0.835707,22
1,m4_switch,vstar,10.057797,9.536232,10.362319,0.835906,4
2,m4_switch,bpsi,-1.145739,-1.362319,-1.072464,0.831101,11
3,m4_switch,bR_switch,2.899948,1.956522,3.550725,0.837042,12
4,m4_switch,s,3.651548,2.427536,4.492754,0.837516,16
5,m4_switch,t,4.553763,4.289855,5.000000,0.827760,11
6,m3_gaussian,k,0.563624,0.313043,0.810145,0.962275,8
7,m3_gaussian,vstar,14.298719,13.666667,15.043478,0.963398,6
8,m3_gaussian,bpsi,-1.943113,-2.000000,-1.710145,0.963474,11
9,m3_gaussian,bR_gaussian,-1.919146,-3.985507,-0.217391,0.962329,27
